# Exchange Rates ETL

Part 1 of the task: pull currency exchange rates from APILayer
(`exchangerates_data/timeseries`) into the MySQL `exchange_rates` table.

## How the load decides what to fetch

There is only one mode, and it covers both the first run and every run after it.
The notebook asks MySQL which dates in the required period are **missing**, then
requests only those.

That replaces the earlier "resume from `MAX(date) + 1 day`" logic, which had a
real defect: if a batch in the middle of the backfill failed, `MAX(date)` had
already moved past the hole, so those days were never requested again. Diffing
against the expected calendar finds a gap wherever it is.

Consecutive missing days are collapsed into windows of at most
`MAX_BATCH_DAYS`, so a cold start issues a handful of year-long calls while a
daily top-up issues exactly one short call. If nothing is missing, **no API
call is made at all** - which matters on a free tier.

## Other implementation choices

- all currencies are requested; the API returns ~172 per day
- history is stored at daily grain, and aggregated to annual by the
  `fact_fx_annual` view rather than in Power BI
- `executemany()` in chunks, not one round-trip per row
- the composite primary key `(date, base_currency, target_currency)` is what
  makes `ON DUPLICATE KEY UPDATE` idempotent
- `executemany()` is chunked, but each API window is committed atomically. A
  failed write therefore cannot leave a date with only some currencies loaded
- transient failures (429 rate limit, 5xx) are retried with exponential
  backoff, honouring `Retry-After` - see `etl.make_session()`
- credentials are read from `.env`, never hard-coded and never printed

`BACKFILL_START_DATE` must stay aligned with `START_YEAR` in the GDP and
population loaders, or the FX years and the indicator years will not line up.

In [1]:
%pip install -q requests python-dotenv mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from datetime import date, timedelta

import etl


# ============================================================
# CONFIGURATION
# ============================================================

# Historical period required for this analytical project.
# Keep in step with START_YEAR in the World Bank loaders.
BACKFILL_START_DATE = date(2020, 1, 1)

# The timeseries endpoint caps a request at roughly one year.
# 365 inclusive days = start date + 364 days.
MAX_BATCH_DAYS = 365

BASE_CURRENCY = "USD"
API_URL = "https://api.apilayer.com/exchangerates_data/timeseries"


# ============================================================
# UPSERT
#
# The aliased-row form (AS new / new.col) rather than VALUES(): MySQL
# deprecated VALUES() inside ON DUPLICATE KEY UPDATE in 8.0.20. Both forms are
# collapsed into a single multi-row statement by mysql-connector's
# executemany - measured on a 5,000-row load, not assumed.
# ============================================================

INSERT_QUERY = """
INSERT INTO exchange_rates
    (`date`, base_currency, target_currency, rate)
VALUES
    (%s, %s, %s, %s) AS new
ON DUPLICATE KEY UPDATE
    rate = new.rate
"""


config = etl.load_config(extra_keys=["API_KEY"])
api_key = config["API_KEY"]

In [3]:
def fetch_window(session, start_date, end_date):
    """
    Request one timeseries window and flatten it into insertable rows.

    The API nests the response as {date: {currency: rate}}; the table is one
    row per (date, base, target).
    """
    payload = etl.request_json(
        session,
        API_URL,
        params={
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "base": BASE_CURRENCY,
        },
        headers={"apikey": api_key},
        timeout=60,
    )

    # APILayer returns HTTP 200 with success=false for quota and plan errors,
    # so the status code alone is not enough to trust the payload.
    if not payload.get("success"):
        raise RuntimeError(f"API returned an unsuccessful response: {payload}")

    base = payload.get("base")
    if base != BASE_CURRENCY:
        raise RuntimeError(
            f"Expected API base {BASE_CURRENCY}, received {base!r}."
        )

    rates = payload.get("rates")
    if not isinstance(rates, dict):
        raise RuntimeError("API response has no rates object.")

    expected_days = {
        (start_date + timedelta(days=offset)).isoformat()
        for offset in range((end_date - start_date).days + 1)
    }
    missing_days = expected_days - set(rates)
    if missing_days:
        sample = ", ".join(sorted(missing_days)[:5])
        raise RuntimeError(f"API omitted {len(missing_days)} requested dates: {sample}")

    return [
        (day, base, target, rate)
        for day, daily_rates in rates.items()
        for target, rate in daily_rates.items()
    ]


def validate_currencies(rows, known_currencies):
    """
    Fail before writing when the provider introduces an unknown currency.

    exchange_rates carries a foreign key to dim_currency. Without this check,
    the first time the provider quotes a new code the load dies on a raw
    constraint error. Loading only the known subset would create incomplete
    dates that a date-level gap detector could no longer recognise.
    """
    unknown = {row[2] for row in rows if row[2] not in known_currencies}
    if unknown:
        codes = ", ".join(sorted(unknown))
        raise RuntimeError(
            f"Provider returned currencies absent from dim_currency: {codes}. "
            "Update the SQL seed before rerunning the window."
        )
    return rows

In [4]:
# ============================================================
# EXTRACT -> TRANSFORM -> LOAD
# ============================================================

session = etl.make_session()
connection = etl.connect_mysql()

try:
    known_currencies = etl.load_currency_keys(connection)

    if not known_currencies:
        raise RuntimeError(
            "dim_currency is empty - run project_1.sql before this notebook."
        )

    # ----------------------------
    # WHAT IS MISSING?
    # ----------------------------
    load_end_date = date.today()

    gaps = etl.missing_dates(
        connection, "exchange_rates", BACKFILL_START_DATE, load_end_date
    )
    windows = etl.to_windows(gaps, MAX_BATCH_DAYS)

    print("Required period:", BACKFILL_START_DATE, "to", load_end_date)
    print("Missing days:", len(gaps))
    print("API calls needed:", len(windows))

    if not windows:
        print("\nExchange rates are already complete. No API call made.")

    total_rows = 0
    for window_start, window_end in windows:
        print(f"\nLoading {window_start} to {window_end}...")

        rows = fetch_window(session, window_start, window_end)
        rows = validate_currencies(rows, known_currencies)

        total_rows += etl.upsert(connection, INSERT_QUERY, rows)
        print(f"Committed {len(rows):,} rows.")

    print(f"\nFinished. Total rows processed: {total_rows:,}")

except Exception:
    print("\nLoad failed. The current API window was rolled back.")
    raise

finally:
    connection.close()
    print("MySQL connection closed.")

Required period: 2020-01-01 to 2026-08-23
Missing days: 0
API calls needed: 0

Exchange rates are already complete. No API call made.

Finished. Total rows processed: 0
MySQL connection closed.


In [5]:
# ============================================================
# VALIDATION
#
# Three things worth checking after every FX load:
#   1. coverage per year, including how many currencies were quoted
#   2. currency-years that do not run to 31 December, which is either a
#      year still in progress or a currency the provider added mid-year
#   3. rates that disagree with their own 31-day neighbourhood, i.e.
#      suspected provider glitches
# ============================================================

connection = etl.connect_mysql()

try:
    print("Coverage by year:")
    etl.print_rows(connection, """
        SELECT
            rate_year                          AS `year`,
            COUNT(*)                           AS rows_loaded,
            COUNT(DISTINCT target_currency)    AS currencies,
            COUNT(DISTINCT `date`)             AS days,
            MIN(`date`)                        AS first_date,
            MAX(`date`)                        AS last_date
        FROM exchange_rates
        GROUP BY rate_year
        ORDER BY rate_year
    """)

    print("Partial currency-years, excluding the year in progress:")
    etl.print_rows(connection, """
        SELECT target_currency, `year`, obs_count, days_in_year, coverage_pct, first_date
        FROM fact_fx_annual
        WHERE is_partial_year = 1
          AND `year` < YEAR(CURRENT_DATE)
        ORDER BY target_currency, `year`
    """)

    print("\nSuspected bad rates, by currency:")
    etl.print_rows(connection, """
        SELECT
            target_currency,
            COUNT(*)     AS suspicious_rows,
            MIN(`date`)  AS first_bad,
            MAX(`date`)  AS last_bad
        FROM fx_rate_outliers
        GROUP BY target_currency
        ORDER BY suspicious_rows DESC
    """)

finally:
    connection.close()

Coverage by year:
   (2020, 60756, 166, 366, datetime.date(2020, 1, 1), datetime.date(2020, 12, 31))
   (2021, 60590, 166, 365, datetime.date(2021, 1, 1), datetime.date(2021, 12, 31))
   (2022, 60702, 169, 365, datetime.date(2022, 1, 1), datetime.date(2022, 12, 31))
   (2023, 62050, 170, 365, datetime.date(2023, 1, 1), datetime.date(2023, 12, 31))
   (2024, 62205, 170, 366, datetime.date(2024, 1, 1), datetime.date(2024, 12, 31))
   (2025, 62384, 172, 365, datetime.date(2025, 1, 1), datetime.date(2025, 12, 31))
   (2026, 40241, 172, 235, datetime.date(2026, 1, 1), datetime.date(2026, 8, 23))
Partial currency-years, excluding the year in progress:
   ('MRU', 2022, 26, 365, Decimal('0.0712'), datetime.date(2022, 12, 6))
   ('SLE', 2022, 53, 365, Decimal('0.1452'), datetime.date(2022, 11, 9))
   ('STN', 2025, 167, 365, Decimal('0.4575'), datetime.date(2025, 7, 18))
   ('VES', 2022, 53, 365, Decimal('0.1452'), datetime.date(2022, 11, 9))
   ('XCG', 2025, 167, 365, Decimal('0.4575'), datetim